# Session 2: the 64-bit format, rounding error and cancellation
**A. Real-world problem.** A physicist computes 1 - cos(angle) for a tiny angle and gets
exactly 0. An engineer solves a quadratic equation and one of the two roots has few correct
digits. Nothing is broken in the computer: the formulas lose digits. We learn how the
64-bit format stores a number, how big the rounding error is, and which formulas to avoid.

**Plan.** (1) decode the 64 bits, (2) measure machine epsilon, (3) see cancellation in the
quadratic formula and in 1 - cos, (4) meet overflow and underflow.

In [ ]:
import pathlib
from decimal import Decimal, getcontext

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: 1 sign bit, 11 exponent bits, 52 fraction bits
A normal 64-bit number is stored as (-1)^s * (1 + f / 2^52) * 2^(e - 1023), where s is the
sign bit, e the 11-bit exponent field (1 to 2046) and f the 52-bit fraction field. The leading
1 is not stored. We read the bits by viewing the same 8 bytes as an integer.

In [ ]:
def decode(x):
    """Return (sign, exponent_field, fraction_field) of the 64-bit float x."""
    bits = int(np.array([x], dtype=np.float64).view(np.uint64)[0])
    return bits >> 63, (bits >> 52) & 0x7FF, bits & ((1 << 52) - 1)


def encode_value(s, e, f):
    """Value of a normal number from its three fields."""
    return (-1.0) ** s * (1.0 + f / 2.0 ** 52) * 2.0 ** (e - 1023)


def machine_epsilon():
    """Gap between 1 and the next float: 2^-52."""
    return float(np.finfo(np.float64).eps)


def rel_error(approx, exact):
    return abs(approx - exact) / abs(exact)

## Step 2: rounding and cancellation
Every operation rounds to the nearest stored number, so the result has a relative error of
at most u = 2^-53 (about 1.1e-16), half of machine epsilon. This is tiny, but it can be
magnified when two nearly equal numbers are subtracted: the leading digits cancel and what
is left is mostly old rounding noise. `small_root_exact` uses 60-digit decimal arithmetic as
a trusted reference.

In [ ]:
getcontext().prec = 60


def small_root_exact(b, c):
    """Smaller root of x^2 - b x + c = 0 (b > 0), in 60-digit decimal arithmetic."""
    B, C = Decimal(float(b)), Decimal(float(c))
    return (B - (B * B - 4 * C).sqrt()) / 2


def roots_naive(b, c):
    """Textbook formula for x^2 - b x + c = 0 with b > 0: (b -/+ sqrt(b^2 - 4c)) / 2."""
    d = np.sqrt(b * b - 4.0 * c)
    return (b + d) / 2.0, (b - d) / 2.0          # (large root, small root)


def roots_stable(b, c):
    """Add the two like-signed terms first, then use x1 * x2 = c for the other root."""
    d = np.sqrt(b * b - 4.0 * c)
    big = (b + d) / 2.0
    return big, c / big


def one_minus_cos_naive(theta):
    return 1.0 - np.cos(theta)


def one_minus_cos_stable(theta):
    """Same quantity from the identity 1 - cos t = 2 sin^2(t / 2): no subtraction."""
    return 2.0 * np.sin(theta / 2.0) ** 2

## Step 3: overflow and underflow
The exponent field limits the range: the largest finite double is about 1.8e308 and the
smallest positive normal one about 2.2e-308. Beyond the top you get `inf` (overflow); below
the bottom, numbers lose digits ("subnormals") and finally become 0 (underflow).

In [ ]:
def float_limits():
    fi = np.finfo(np.float64)
    return float(fi.max), float(fi.tiny), float(np.nextafter(0.0, 1.0))


def length_naive(x, y):
    with np.errstate(over="ignore"):
        return float(np.sqrt(x * x + y * y))


def length_safe(x, y):
    """Scale by the larger value first so that no square overflows."""
    m = max(abs(x), abs(y))
    if m == 0.0:
        return 0.0
    return float(m * np.sqrt((x / m) ** 2 + (y / m) ** 2))

## Step 4: see it
Left: relative error of the small root of x^2 - b x + 1 = 0 for the naive and stable formulas
as b grows. Right: the error of 1 - cos for small angles.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for x in (1.0, -6.5, 0.1):
        s, e, f = decode(x)
        print(f"{x:5}: sign {s}, exponent field {e} (true exponent {e - 1023}), fraction {f:013x} hex")
    print("machine epsilon (numpy):", machine_epsilon())
    for b in (1e2, 1e4, 1e6, 1e8):
        ex = float(small_root_exact(b, 1.0))
        print(f"b={b:.0e}: naive error {rel_error(roots_naive(b, 1.0)[1], ex):.1e}, "
              f"stable error {rel_error(roots_stable(b, 1.0)[1], ex):.1e}")
    for t in (1e-2, 1e-6, 1e-8):
        print(f"theta={t:.0e}: naive 1-cos {one_minus_cos_naive(t):.4e}, stable {one_minus_cos_stable(t):.4e}")
    print("limits (max, smallest normal, smallest subnormal):", float_limits())
    print("length of (3e200, 4e200): naive", length_naive(3e200, 4e200), ", safe", length_safe(3e200, 4e200))

    bs = np.logspace(0.5, 8, 60)
    en = np.array([max(rel_error(roots_naive(b, 1.0)[1], float(small_root_exact(b, 1.0))), 1e-17) for b in bs])
    es = np.array([max(rel_error(roots_stable(b, 1.0)[1], float(small_root_exact(b, 1.0))), 1e-17) for b in bs])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(bs, en, label="textbook formula")
    ax.loglog(bs, es, label="stable formula")
    ax.axhline(2.0 ** -53, color="gray", ls=":", label="u = 2^-53")
    ax.set_xlabel("coefficient b (no unit)")
    ax.set_ylabel("relative error of the small root (floor 1e-17)")
    ax.set_title("Small root of x^2 - b x + 1 = 0")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "quadratic_error.png", dpi=120)

    th = np.logspace(-9, -1, 60)
    ref = 2 * np.sin(th / 2) ** 2
    err = np.maximum(np.abs(one_minus_cos_naive(th) - ref) / ref, 1e-17)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.loglog(th, err, label="1 - cos(theta)")
    ax2.set_xlabel("angle theta (radians)")
    ax2.set_ylabel("relative error (floor 1e-17)")
    ax2.set_title("Subtracting nearly equal numbers destroys digits")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "one_minus_cos_error.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Decode 1.0, 2.0 and 0.5 with `decode` and describe the pattern of the exponent field.
2. **Stretch.** Add to the lesson a function `roots_stable_general(a, b, c)` for any signs of b, using q = -(b + sign(b) sqrt(b^2 - 4ac)) / 2, x1 = q / a, x2 = c / q, and test it on a = 1, b = -1e8, c = 1.
3. **Explain.** Explain why the stable quadratic formula computes the second root as c / x1 instead of subtracting, using Vieta's relation x1 x2 = c.

## Assessment
- **A. Conceptual.** Why does computing 1 - cos(1e-8) give exactly 0 although each single operation is accurate to 16 digits?
- **B. Numerical.** Decode 0.15625 (that is 5/32): give s, e and the fraction field f. Sanity check by rebuilding the value.
- **C. Graph and simulation interpretation.** From the app graph view (c = 1), roughly how does the error of the textbook small root change when p increases by 1? About how many digits are correct at p = 6?
- **D. Design and optimisation challenge.** Design: a program must evaluate 1 - cos(t) for every angle from 1e-9 to 1 radian with a relative error below 1e-12. Choose the formula(s). Where, if anywhere, is the textbook form acceptable?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("machine epsilon equals 2^-52 (about 2.22e-16)", L.machine_epsilon() == 2.0 ** -52))
    s, e, f = L.decode(-6.5)
    checks.append(("worked example: -6.5 has sign 1, exponent field 1025, fraction 0.625 * 2^52",
                   (s, e, f) == (1, 1025, int(0.625 * 2 ** 52))))
    checks.append(("decoded fields rebuild the value of 0.1 exactly", L.encode_value(*L.decode(0.1)) == 0.1))
    checks.append(("hand calculation: 0.15625 = 1.25 * 2^-3 has fields (0, 1020, 2^50)", L.decode(0.15625) == (0, 1020, 2 ** 50)))
    checks.append(("1.0 has exponent field 1023 and fraction 0", L.decode(1.0) == (0, 1023, 0)))
    ex = float(L.small_root_exact(1e6, 1.0))
    checks.append(("worked example: small root of x^2 - 1e6 x + 1 is 1.000000000001e-6", abs(ex - 1.000000000001e-6) < 1e-18))
    en = L.rel_error(L.roots_naive(1e6, 1.0)[1], ex)
    es = L.rel_error(L.roots_stable(1e6, 1.0)[1], ex)
    checks.append(("worked example: textbook formula loses about 5 digits at b = 1e6 (error 1e-6 to 1e-5)", 1e-6 < en < 1e-5))
    checks.append(("stable formula is accurate to 1e-15 for b = 1e2 ... 1e8",
                   all(L.rel_error(L.roots_stable(b, 1.0)[1], float(L.small_root_exact(b, 1.0))) < 1e-15 for b in (1e2, 1e4, 1e6, 1e8))))
    checks.append(("hand calculation: at b = 1e8 the textbook small root is off by more than 10 percent",
                   L.rel_error(L.roots_naive(1e8, 1.0)[1], float(L.small_root_exact(1e8, 1.0))) > 0.1))
    checks.append(("both formulas agree on the large root", np.isclose(L.roots_naive(1e3, 1.0)[0], L.roots_stable(1e3, 1.0)[0], rtol=1e-15)))
    checks.append(("edge: 1 - cos(1e-8) is exactly 0 but 2 sin^2(t/2) is 5e-17",
                   L.one_minus_cos_naive(1e-8) == 0.0 and abs(L.one_minus_cos_stable(1e-8) - 5e-17) < 1e-20))
    checks.append(("1 - cos(0.5) is the same for both formulas to 1e-15",
                   abs(L.one_minus_cos_naive(0.5) - L.one_minus_cos_stable(0.5)) < 1e-15))
    big, tiny, sub = L.float_limits()
    checks.append(("limits: max about 1.8e308, smallest normal about 2.2e-308, smallest subnormal 5e-324",
                   1.79e308 < big < 1.80e308 and 2.2e-308 < tiny < 2.3e-308 and sub == 5e-324))
    checks.append(("edge: naive length of (3e200, 4e200) overflows, safe length is 5e200",
                   np.isinf(L.length_naive(3e200, 4e200)) and abs(L.length_safe(3e200, 4e200) / 5e200 - 1) < 1e-15))
    checks.append(("edge: safe length of (0, 0) is 0 and of (3, 4) is 5", L.length_safe(0.0, 0.0) == 0.0 and abs(L.length_safe(3.0, 4.0) - 5.0) < 1e-15))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
